# Systemy agentów

## 1. Wprowadzenie

**Agent** to system oparty na modelu językowym, który obserwuje otoczenie (np. dane, narzędzia, historię rozmowy), samodzielnie planuje kolejne kroki i wywołuje dostępne narzędzia, aby zrealizować cel użytkownika.

**Agentic RAG** to podejście, w którym agent nie tylko pobiera dokumenty (jak w klasycznym RAG), ale aktywnie decyduje, kiedy i jak je wyszukiwać, łączyć, filtrować oraz ewentualnie wykonywać dodatkowe kroki (np. dekompozycję zadania) przed wygenerowaniem odpowiedzi.

### Cel ćwiczenia
Zbudujemy agenta, który: na podstawie problemu zdrowotnego X, lokalizacji Y i innych preferencji uzytkownika zaproponuje przychodnię, korzystając z danych strukturalnych + opinii pacjentów.

## 2. Przygotowanie danych

In [1]:
!pip install langgraph langchain-openai langchain-core
!pip install --quiet "transformers[torch]" accelerate sentencepiece safetensors
!pip install "langchain>=0.3" "langchain-community>=0.3" "langgraph>=1.0"
!pip install "faiss-cpu>=1.7" "sentence-transformers>=3.0"
!pip install "transformers[torch]>=4.45" "accelerate>=0.34" "safetensors>=0.4"
!pip install torch>=2.3"

/bin/bash: -c: line 1: unexpected EOF while looking for matching `"'
/bin/bash: -c: line 2: syntax error: unexpected end of file


In [2]:
import pandas as pd

In [3]:
clinics_df = pd.read_csv("clinics.csv")
clinics_df.head()

,clinic_id,name,city,county,voivodeship,address,diab,endo,cardio,pulm,nephro,rating,rating_count
0,1,"NIEPUBLICZNY ZAKŁAD OPIEKI ZDROWOTNEJ ""SOL-MED...",WARSZAWA,M. St. Warszawa,Mazowieckie,AL. 3 MAJA 2/35,0,0,0,1,0,4.2,15.0
1,2,"CENTRALNA WOJSKOWA PRZYCHODNIA LEKARSKA ""CEPEL...",WARSZAWA,M. St. Warszawa,Mazowieckie,KOSZYKOWA 78,0,0,1,0,0,2.5,581.0
2,3,"LECZNICA ""GOCŁAW"" MARIA KAMELA - PORADNIA LEKA...",WARSZAWA,M. St. Warszawa,Mazowieckie,AFRYKAŃSKA 7A,0,0,1,0,0,4.0,27.0
3,4,LUX MED ONKOLOGIA SPÓŁKA Z OGRANICZONĄ ODPOWIE...,WARSZAWA,M. St. Warszawa,Mazowieckie,"GEN. AUGUSTA EMILA FIELDORFA ""NILA"" 40",0,0,1,0,0,4.0,370.0
4,5,NIEPUBLICZNY ZAKŁAD OPIEKI ZDROWOTNEJ LECZNICA...,WARSZAWA,M. St. Warszawa,Mazowieckie,KLECZEWSKA 41,0,0,1,0,0,4.1,101.0


In [4]:
reviews_df = pd.read_csv("clinic_reviews.csv")
reviews_df.head()

,clinic_id,review_id,rating,text
0,1,592b0f5a0e7e,1,"Brak możliwości zamówienia e-recepty online, a..."
1,1,89f48caae50f,2,"Cóż, dziwne miejsce w porównaniu do tych ,któr..."
2,1,36b5a47ad396,5,"Pani dr.Iwona Ģąsiewska bardzo miła,empatyczna..."
3,1,086485f820e2,5,Bardzo dobra przychodnia. Nie ma problemu z do...
4,1,aeb90042392b,4,w przypadku niewielkich dolegliwości i tylko t...


## 3. Integracja LangChain + LangGraph z lokalnym modelem


In [5]:
import transformers
import torch

# model_name = "CYFRAGOVPL/Llama-PLLuM-8B-instruct"
# opcjonalnie
model_name = "Qwen/Qwen2.5-3B-Instruct"
# model_name = "speakleash/Bielik-4.5B-v3.0-Instruct" # należy zarejestrować konto w Hugging Face, zatwierdzić warunki licencji na stronie Bielika i wgrać token dostępowy

prepared_prompt = "Szukam przychodni w Warszawie. Bolą mnie oczy. Jaka jest najbliższa przychodnia?"

pipeline = transformers.pipeline(
    "text-generation",
    model=model_name,
    model_kwargs={"torch_dtype": torch.float16}, #float16 if not ampere/hopper gpu
    device_map="auto",
)

messages = [
    {"role": "system", "content": "You are a clinics advisor chatbot assistant!"},
    {"role": "user", "content": prepared_prompt},
]

outputs = pipeline(
    messages,
    max_new_tokens=256,
)
print(outputs[0]["generated_text"][-1])

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


{'role': 'assistant', 'content': 'Przykro mi, ale nie mogę podać konkretnych informacji o najbliższej teraz przychodni ocznej w Warszawie bez dodatkowych szczegółów. Możesz spróbować znaleźć informacje na następujące sposoby:\n\n1. Skonsultuj się z lekarzem telewizyjnym poprzez aplikację lub telefon.\n2. Sprawdź strony internetowe niektórych terenowych klinik ocznych w Warszawie.\n3. Skorzystaj z aplikacji Google Maps, wprowadzając "terenowe kliniki oczne w Warszawie" i zaznaczając "Najbliższe".\n4. Zapytaj lokalnego doradcę lub znajomego.\n\nJeśli jesteś w trakcie podróży i ból oczny jest poważny, zalecam skorzystanie z pierwszej pomocy lub wyjście do najbliższego szpitala.'}


In [6]:
import math
import json
from typing import Any, Dict, List, TypedDict

import pandas as pd
from langchain_classic.chains.llm import LLMChain
from langchain_core.prompts import PromptTemplate
from langchain_core.documents import Document
from langchain_core.tools import StructuredTool, Tool
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.llms import HuggingFacePipeline
from langchain_community.vectorstores import FAISS
from langgraph.graph import END, StateGraph

# Reuse the transformers pipeline prepared above
llm = HuggingFacePipeline(pipeline=pipeline)

# Short LLM test via LangChain
quick_prompt = PromptTemplate.from_template("Podaj krótką poradę dla pacjenta z bólem głowy w Warszawie - gdzie ma się udać?")
quick_chain = LLMChain(llm=llm, prompt=quick_prompt)
_ = quick_chain.invoke({})
print("Test LangChain + lokalny model działa (odpowiedź przycięta)")


/tmp/ipykernel_55949/3194986239.py:10: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import HuggingFaceEmbeddings
/tmp/ipykernel_55949/3194986239.py:16: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=pipeline)
/tmp/ipykernel_55949/3194986239.py:20: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 2.0.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  quic

Test LangChain + lokalny model działa (odpowiedź przycięta)


## 4. Przygotowanie danych i prosty filtr kandydatów


In [7]:
clinics_df["rating"] = pd.to_numeric(clinics_df["rating"], errors="coerce").fillna(0.0)
clinics_df["rating_count"] = pd.to_numeric(clinics_df["rating_count"], errors="coerce").fillna(0)

In [8]:

# Mapowanie problem -> kolumny specjalizacji w danych strukturalnych
disease_to_specialty = {
    "cukrzyca": ["diab"],
    "diabet": ["diab"],
    "tarczy": ["endo"],
    "endo": ["endo"],
    "serc": ["cardio"],
    "kardio": ["cardio"],
    "płu": ["pulm"],
    "astm": ["pulm"],
    "nefro": ["nephro"],
    "nerk": ["nephro"],
}


def detect_specialties(problem: str) -> List[str]:
    text = (problem or "").lower()
    hits = []
    for keyword, cols in disease_to_specialty.items():
        if keyword in text:
            hits.extend(cols)
    return list(dict.fromkeys(hits))  # dedupe while keeping order


In [9]:
def filter_clinics(problem: str, location: str = "", preferences: Dict[str, Any] | None = None, top_k: int = 30) -> List[Dict[str, Any]]:
    prefs = preferences or {}
    min_rating = float(prefs.get("min_rating", 0))
    min_count = int(prefs.get("min_rating_count", 0))

    specialties = detect_specialties(problem)
    df = clinics_df.copy()
    df = df[df["city"].str.upper() == "WARSZAWA"]

    if location:
        loc = location.upper()
        mask_loc = df["address"].str.upper().str.contains(loc) | df["name"].str.upper().str.contains(loc)
        df = df[mask_loc]

    if specialties:
        spec_mask = False
        for spec in specialties:
            spec_mask = spec_mask | (df[spec] == 1)
        df = df[spec_mask]

    df = df[(df["rating"] >= min_rating) & (df["rating_count"] >= min_count)]

    df = df.sort_values(by=["rating", "rating_count"], ascending=[False, False]).head(top_k)

    return [
        {
            "clinic_id": int(row.clinic_id),
            "name": row.name,
            "address": row.address,
            "rating": float(row.rating),
            "rating_count": int(row.rating_count),
            "specialties": specialties or "any",
        }
        for row in df.itertuples()
    ]


# Szybki test filtrowania
filter_clinics("problemy z sercem", "URSYNÓW", {"min_rating": 3.5, "min_rating_count": 50})[:3]


[{'clinic_id': 36,
  'name': 'SAMODZIELNY PUBLICZNY ZAKŁAD OPIEKI ZDROWOTNEJ WARSZAWA-URSYNÓW - PORADNIA PODSTAWOWEJ OPIEKI ZDROWOTNEJ',
  'address': 'KŁOBUCKA 14',
  'rating': 4.0,
  'rating_count': 55,
  'specialties': ['cardio']}]

In [10]:
## 5. Opinie pacjentów: embeddingi i retriever

reviews_df["rating"] = pd.to_numeric(reviews_df["rating"], errors="coerce").fillna(0.0)

review_docs: List[Document] = [
    Document(
        page_content=str(row.text),
        metadata={
            "clinic_id": int(row.clinic_id),
            "review_id": row.review_id,
            "rating": float(row.rating),
        },
    )
    for row in reviews_df.itertuples()
]

embedding_model_name = "sdadas/mmlw-retrieval-roberta-large"
embeddings = HuggingFaceEmbeddings(model_name=embedding_model_name)

vectorstore = FAISS.from_documents(review_docs, embeddings)
reviews_retriever = vectorstore.as_retriever(search_kwargs={"k": 50})

print(f"Załadowano {len(review_docs)} opinii, gotowy retriever RAG")


/tmp/ipykernel_55949/4245684492.py:18: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name=embedding_model_name)


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Załadowano 994 opinii, gotowy retriever RAG


In [11]:
## 6. Narzędzia agenta (StructuredTool / Tool)


def filter_clinics_tool(problem: str, location: str = "", preferences: Dict[str, Any] | None = None) -> List[Dict[str, Any]]:
    return filter_clinics(problem, location, preferences or {}, top_k=30)


def retrieve_reviews_tool(problem: str, clinic_ids: List[int], k_per_clinic: int = 3) -> List[Dict[str, Any]]:
    """Select top opinions per clinic using similarity search and metadata filter."""
    if not clinic_ids:
        return []

    retrieved = reviews_retriever.invoke(problem)
    bucket: Dict[int, List[Document]] = {}
    for doc in retrieved:
        cid = int(doc.metadata.get("clinic_id", -1))
        if cid not in clinic_ids:
            continue
        if len(bucket.get(cid, [])) >= k_per_clinic:
            continue
        bucket.setdefault(cid, []).append(doc)

    flat: List[Dict[str, Any]] = []
    for cid, docs in bucket.items():
        for doc in docs:
            flat.append(
                {
                    "clinic_id": cid,
                    "review_id": doc.metadata.get("review_id"),
                    "rating": float(doc.metadata.get("rating", 0)),
                    "text": doc.page_content,
                }
            )
    return flat


def rank_clinics(candidates: List[Dict[str, Any]], reviews: List[Dict[str, Any]], top_k: int = 5) -> List[Dict[str, Any]]:
    if not candidates:
        return []

    review_score: Dict[int, float] = {}
    review_count: Dict[int, int] = {}
    for r in reviews:
        cid = int(r["clinic_id"])
        review_score[cid] = review_score.get(cid, 0.0) + float(r.get("rating", 0))
        review_count[cid] = review_count.get(cid, 0) + 1

    ranked = []
    for c in candidates:
        cid = c["clinic_id"]
        avg_review = review_score.get(cid, 0.0) / review_count.get(cid, 1)
        score = 0.6 * c["rating"] + 0.2 * math.log1p(c["rating_count"]) + 0.2 * avg_review
        ranked.append({"score": score, **c})

    ranked = sorted(ranked, key=lambda x: x["score"], reverse=True)[:top_k]
    return ranked


filter_clinics_structured = StructuredTool.from_function(
    name="filter_clinics_tool",
    func=filter_clinics_tool,
    description="Filtruje listę przychodni w Warszawie po specjalizacji, lokalizacji i ratingach."
)

retrieve_reviews = Tool(
    name="retrieve_reviews_tool",
    func=lambda query, clinic_ids: retrieve_reviews_tool(query, clinic_ids),
    description="Zwraca trafne opinie pacjentów dla listy clinic_ids na podstawie zapytania."
)

print("Narzędzia zbudowane: filter_clinics_tool, retrieve_reviews_tool, rank_clinics")


Narzędzia zbudowane: filter_clinics_tool, retrieve_reviews_tool, rank_clinics


In [12]:
## 7. Graf LangGraph: parse -> search -> reviews -> ranking -> answer

class AgentState(TypedDict):
    user_query: str
    parsed_request: Dict[str, Any]
    candidates: List[Dict[str, Any]]
    retrieved_reviews: List[Dict[str, Any]]
    ranked_clinics: List[Dict[str, Any]]
    answer: str


parse_prompt = PromptTemplate.from_template(
    """
    Z prośby użytkownika wyodrębnij:
    - problem zdrowotny (problem)
    - lokalizację (location, Warszawa jeśli brak)
    - preferencje (preferences: min_rating, min_rating_count jeśli są)

    Zwrot JSON bez komentarzy o kluczach: problem, location, preferences.
    Prośba: {user_query}
    """
)
parse_chain = LLMChain(llm=llm, prompt=parse_prompt)


def parse_request_node(state: AgentState) -> Dict[str, Any]:
    raw = parse_chain.invoke({"user_query": state["user_query"]})
    try:
        parsed = json.loads(raw["text"] if isinstance(raw, dict) else str(raw))
    except Exception:
        parsed = {
            "problem": state.get("user_query", "problem zdrowotny"),
            "location": "Warszawa",
            "preferences": {},
        }
    return {"parsed_request": parsed}


def structured_search_node(state: AgentState) -> Dict[str, Any]:
    req = state.get("parsed_request", {})
    candidates = filter_clinics(
        req.get("problem", ""),
        req.get("location", ""),
        req.get("preferences", {}),
        top_k=30,
    )
    return {"candidates": candidates}


def reviews_rag_node(state: AgentState) -> Dict[str, Any]:
    req = state.get("parsed_request", {})
    clinic_ids = [c["clinic_id"] for c in state.get("candidates", [])][:20]
    reviews = retrieve_reviews_tool(req.get("problem", ""), clinic_ids, k_per_clinic=3)
    return {"retrieved_reviews": reviews}


def ranking_node(state: AgentState) -> Dict[str, Any]:
    ranked = rank_clinics(state.get("candidates", []), state.get("retrieved_reviews", []), top_k=5)
    return {"ranked_clinics": ranked}


answer_prompt = PromptTemplate.from_template(
    """
    Jesteś asystentem rekomendującym przychodnie POZ w Warszawie.
    Użyj kandydatów i opinii, aby zaproponować maks 3 najlepsze opcje.

    Problem: {problem}
    Lokalizacja: {location}
    Preferencje: {preferences}

    Kandydaci (JSON): {candidates}
    Opinie (JSON): {reviews}

    Zwróć listę rekomendacji w języku polskim:
    - nazwa przychodni + adres
    - dlaczego pasuje (specjalizacja, ocena, liczba opinii)
    - krótki cytat z opinii jeśli dostępny
    """
)
answer_chain = LLMChain(llm=llm, prompt=answer_prompt)


def answer_node(state: AgentState) -> Dict[str, Any]:
    req = state.get("parsed_request", {})
    text = answer_chain.invoke(
        {
            "problem": req.get("problem", ""),
            "location": req.get("location", ""),
            "preferences": req.get("preferences", {}),
            "candidates": state.get("ranked_clinics", []),
            "reviews": state.get("retrieved_reviews", []),
        }
    )
    content = text["text"] if isinstance(text, dict) else str(text)
    return {"answer": content}


builder = StateGraph(AgentState)
builder.add_node("parse_request", parse_request_node)
builder.add_node("structured_search", structured_search_node)
builder.add_node("reviews_rag", reviews_rag_node)
builder.add_node("ranking", ranking_node)
builder.add_node("answer", answer_node)

builder.set_entry_point("parse_request")
builder.add_edge("parse_request", "structured_search")
builder.add_edge("structured_search", "reviews_rag")
builder.add_edge("reviews_rag", "ranking")
builder.add_edge("ranking", "answer")
builder.add_edge("answer", END)

app = builder.compile()
print("Graf LangGraph skompilowany.")


Graf LangGraph skompilowany.


In [13]:
## 8. Funkcja pomocnicza run_agent + przykładowe zapytania


def run_agent(query: str) -> Dict[str, Any]:
    state = app.invoke({"user_query": query})
    return {
        "answer": state.get("answer", ""),
        "ranked": state.get("ranked_clinics", []),
        "reviews": state.get("retrieved_reviews", []),
    }


examples = [
    "Mieszkam na Ursynowie, mam cukrzycę i chcę dobrą ocenę oraz dużo opinii.",
    "Szukam czegoś przy Grójeckiej, mam problemy z sercem i zależy mi na dobrych opiniach.",
]

for q in examples:
    print("\n### Zapytanie:", q)
    result = run_agent(q)
    print(result["answer"][:800])
    print("\nTop kandydaci (skrócone):")
    for c in result["ranked"][:3]:
        print({k: c[k] for k in ["clinic_id", "name", "rating", "rating_count", "score"]})


[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



### Zapytanie: Mieszkam na Ursynowie, mam cukrzycę i chcę dobrą ocenę oraz dużo opinii.


[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



    Jesteś asystentem rekomendującym przychodnie POZ w Warszawie.
    Użyj kandydatów i opinii, aby zaproponować maks 3 najlepsze opcje.

    Problem: Mieszkam na Ursynowie, mam cukrzycę i chcę dobrą ocenę oraz dużo opinii.
    Lokalizacja: Warszawa
    Preferencje: {}

    Kandydaci (JSON): [{'score': 4.4021747728346625, 'clinic_id': 197, 'name': 'ZAKŁAD OPIEKI ZDROWOTNEJ WARSZAWA BEMOWO-WŁOCHY - PORADNIA LEKARZA POZ', 'address': 'JANISZOWSKA 15', 'rating': 4.3, 'rating_count': 60, 'specialties': 'any'}, {'score': 4.181610040220442, 'clinic_id': 151, 'name': 'SAMODZIELNY  ZESPÓŁ PUBLICZNYCH  ZAKŁADÓW  LECZNICTWA OTWARTEGO WARSZAWA WESOŁA - PORADNIA LEKARZ PODSTAWOWEJ OPIEKI ZDROWOTNEJ', 'address': 'KAMYK 10A', 'rating': 4.4, 'rating_count': 14, 'specialties': 'any'}, {'score': 3.72225343

Top kandydaci (skrócone):
{'clinic_id': 197, 'name': 'ZAKŁAD OPIEKI ZDROWOTNEJ WARSZAWA BEMOWO-WŁOCHY - PORADNIA LEKARZA POZ', 'rating': 4.3, 'rating_count': 60, 'score': 4.4021747728346625}
{'clini

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



    Jesteś asystentem rekomendującym przychodnie POZ w Warszawie.
    Użyj kandydatów i opinii, aby zaproponować maks 3 najlepsze opcje.

    Problem: Szukam czegoś przy Grójeckiej, mam problemy z sercem i zależy mi na dobrych opiniach.
    Lokalizacja: Warszawa
    Preferencje: {}

    Kandydaci (JSON): [{'score': 3.9610340371976185, 'clinic_id': 170, 'name': 'SAMODZIELNY ZESPÓŁ PUBLICZNYCH ZAKŁADÓW LECZNICTWA OTWARTEGO WARSZAWA OCHOTA - PORADNIA LEKARZA POZ', 'address': 'SKARŻYŃSKIEGO 1', 'rating': 3.4, 'rating_count': 99, 'specialties': ['cardio']}, {'score': 3.9180743515792327, 'clinic_id': 199, 'name': 'ZAKŁAD OPIEKI ZDROWOTNEJ WARSZAWA BEMOWO-WŁOCHY - PORADNIA LEKARZA POZ', 'address': 'SZYBOWCOWA 4', 'rating': 3.9, 'rating_count': 17, 'specialties': ['cardio']}, {'score': 3.72225343

Top kandydaci (skrócone):
{'clinic_id': 170, 'name': 'SAMODZIELNY ZESPÓŁ PUBLICZNYCH ZAKŁADÓW LECZNICTWA OTWARTEGO WARSZAWA OCHOTA - PORADNIA LEKARZA POZ', 'rating': 3.4, 'rating_count': 99, 'score'

## 9. Ograniczenia i możliwe rozszerzenia
- Brak geolokalizacji – filtr po adresie/dzielnicy jest tekstowy.
- Heurystyczne mapowanie problemów na specjalizacje; warto rozbudować słownik lub użyć klasyfikatora.
- Ranking korzysta z prostych wag; można dodać sentyment opinii lub model oceny jakości.
- Opinie są indeksowane jedynie dla wyszukiwania semantycznego; brak filtrowania po świeżości opinii.
- Można dodać kolejkowanie narzędzi (np. sprawdzanie dostępności terminów) i lepszą obsługę błędów parsowania JSON w węźle `parse_request`.



---

# ZADANIE DO WYKONANIA - do wyboru:

- poniższe zadanie ( węzeł filtrowania jakości do grafu agenta)

lub

- wykonać jedno z powyższych ograniczeń/rozszerzeń


## Dodaj węzeł filtrowania jakości do grafu agenta

### Zadanie:

Obecny agent zwraca wszystkie przychodnie bez dodatkowej walidacji jakości.
Twoim zadaniem jest **dodać nowy węzeł do grafu LangGraph**, który:

1. **Sprawdza jakość kandydatów** po wyszukiwaniu strukturalnym
2. **Odrzuca przychodnie niskiej jakości**:
   - Rating < 3.5 (za niski)
   - Liczba opinii < 10 (za mało danych)
   - Brak opinii w RAG (nie można zweryfikować)
3. ** Jeśli zostało \< 3 kandydatów ** → rozluźnia kryteria (rating >= 3.0)
4. **Loguje** co odrzucił i dlaczego

### Wymagania:

**Krok 1:** Napisz funkcję węzła `quality_filter_node`

```python
def quality_filter_node(state: AgentState) -> Dict[str, Any]:
    """
    Filtruje kandydatów po jakości (rating, liczba opinii).
    Jeśli zostaje < 3 kandydatów, rozluźnia kryteria.
    """
    candidates = state.get("candidates", [])
    
    # TODO: Implementuj logikę filtrowania
    # 1. Filtruj po rating: np.  >= 3.5 i rating_count >= 10
    # 2. Jeśli zostało < 3 → użyj rating np. >= 3.0
    # 3. Loguj odrzucone
    
    return {"candidates": filtered_candidates}
```

**Krok 2:** Dodaj węzeł do grafu

```python
builder.add_node(...)
(...)
```

**Krok 3:** Zmień przepływ grafu

```python
builder.add_edge(...)
(...)

```


### Oczekiwany przykładowy output:

```
🔍 Quality Filter: Otrzymano 30 kandydatów
 Tylko 2 kandydatów - rozluźniam kryteria
   ✓ Przywrócono: PRZYCHODNIA FAMILIA MOKOTÓW...
   ✓ Przywrócono: CENTRUM MEDYCZNE URSYNÓW...
 Zaakceptowano: 5 przychodni
 Odrzucono: 25 przychodni (niski rating lub za mało opinii)
```

### Wizualizacja nowego grafu:

```
parse_request
    ↓
structured_search (30 kandydatów)
    ↓
quality_filter    (filtruje do 5 kandydatów)  
    ↓
reviews_rag       (pobiera opinie dla 5)
    ↓
ranking
    ↓
answer
```


### Kryteria:

-  Funkcja `quality_filter_node` poprawnie filtruje kandydatów
-  Logika rozluźniania kryteriów działa (< 3 kandydatów)
-  Węzeł jest dodany do grafu w odpowiednim miejscu
-  Graf jest poprawnie przebudowany
-  Logi pokazują co węzeł robi
-  Agent działa i zwraca tylko wysokiej jakości przychodnie

---

Ten węzeł pokazuje **kluczową koncepcję w systemach agentów**:
- **Walidacja pośrednia** - sprawdzanie jakości między krokami
- **Adaptacja** - dynamiczne dostosowanie kryteriów
- **Obserwability** - logowanie do debugowania
- **Modularność** - łatwo dodać/usunąć węzeł z grafu

In [14]:
from typing import Dict, Any

def quality_filter_node(state: AgentState) -> Dict[str, Any]:
    """
    Filtruje kandydatów według kryteriów jakości.
    Jeśli zostaje < 3 kandydatów, rozluźnia kryteria.
    """
    MIN_RATING_STRICT = 3.5
    MIN_RATING_RELAXED = 3.0
    MIN_COUNT_STRICT = 10

    # pobierz kandydatów ze stanu
    candidates = state.get("candidates", [])
    print(f"\n🔍 Quality Filter: Otrzymano {len(candidates)} kandydatów")

    # KROK 1: Pierwsze filtrowanie (rygorystyczne)
    review_counts = reviews_df.groupby("clinic_id").size()
    filtered = []
    rejected = []
    for clinic in candidates:
        reasons = []
        if clinic["rating"] < MIN_RATING_STRICT:
            reasons.append(
                f"rating {clinic['rating']:.1f} < {MIN_RATING_STRICT}"
            )
        if clinic["rating_count"] < MIN_COUNT_STRICT:
            reasons.append(
                f"liczba opinii {clinic['rating_count']} < {MIN_COUNT_STRICT}"
            )
        if review_counts.get(clinic["clinic_id"], default=0)==0:
            reasons.append("brak opinii w RAG")
        if reasons:
            rejected.append((clinic, reasons))
        else:
            filtered.append(clinic)
    print(f"Po rygorystycznym filtrowaniu: {len(filtered)} kandydatów")

    # KROK 2: Jeśli za mało, rozluźnij kryteria
    if len(filtered) < 3:
        print(f"Tylko {len(filtered)} kandydatów - rozluźniam kryteria")
        for (clinic, reasons) in rejected:
            if (
                clinic["rating"] >= MIN_RATING_RELAXED
                and clinic["rating_count"] >= MIN_COUNT_STRICT
            ):
                filtered.append(clinic)
                rejected.remove((clinic, reasons))
                print(
                f"\t✓ Przywrócono: {clinic['name'][:50]:50} "
                f"(rating={clinic['rating']:.1f})"
                )

    # KROK 3: Logowanie końcowe
    print("\nOdrzucone przychodnie:")
    for clinic, reasons in rejected:
        print(
            f"\t✗ {clinic['name'][:50]:50} -> "
            + ", ".join(reasons)
        )
    print(f"\n✓ Zaakceptowano: {len(filtered)} przychodni")
    print(f"✗ Odrzucono: {len(candidates) - len(filtered)} przychodni")

    # Jeśli nadal 0 kandydatów, zwróć oryginalne (fallback)
    if len(filtered) == 0:
        print("UWAGA: Żaden kandydat nie spełnia kryteriów - zwracam wszystkich")
        filtered = candidates

    return {"candidates": filtered}

In [15]:
from langgraph.graph import StateGraph, END

# Przebuduj graf
builder = StateGraph(AgentState)

# Dodaj wszystkie węzły
builder.add_node("parse_request", parse_request_node)
builder.add_node("structured_search", structured_search_node)
builder.add_node("quality_filter", quality_filter_node)
builder.add_node("reviews_rag", reviews_rag_node)
builder.add_node("ranking", ranking_node)
builder.add_node("answer", answer_node)

# Definiuj przepływ
builder.set_entry_point("parse_request")
builder.add_edge("parse_request", "structured_search")
builder.add_edge("structured_search", "quality_filter")
builder.add_edge("quality_filter", "reviews_rag")
builder.add_edge("reviews_rag", "ranking")
builder.add_edge("ranking", "answer")
builder.add_edge("answer", END)

# Skompiluj NOWY graf
app = builder.compile()

print("Nowy graf z quality_filter skompilowany")

Nowy graf z quality_filter skompilowany


In [16]:
# przykład

query1 = "Szukam przychodni na Ursynowie z diabetologiem"
result1 = run_agent(query1)

print(f"\nZnaleziono: {len(result1['ranked'])} przychodni")
for i, clinic in enumerate(result1['ranked'][:3], 1):
    print(f"{i}. {clinic['name'][:50]} - Rating: {clinic['rating']:.1f}/5.0")

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



🔍 Quality Filter: Otrzymano 30 kandydatów
Po rygorystycznym filtrowaniu: 14 kandydatów

Odrzucone przychodnie:
	✗ SAMODZIELNY ZESPÓŁ PUBLICZNYCH ZAKŁADÓW LECZNICTWA -> rating 3.4 < 3.5
	✗ SAMODZIELNY PUBL ZESP ZAKŁ LECZNICTWA OTWARTEGO WA -> rating 3.4 < 3.5
	✗ GRUPA ZDROWIE WARSZAWA SZYMANOWSKIEGO SPÓŁKA Z OGR -> rating 3.4 < 3.5
	✗ SAMODZIELNY PUBL ZESP ZAKŁ LECZNICTWA OTWARTEGO WA -> rating 3.3 < 3.5
	✗ CENTERMED WARSZAWA SP Z O. O - PORADNIA LEKARZA PO -> rating 3.3 < 3.5
	✗ SAMODZIELNY ZESPÓŁ PUBLICZNYCH ZAKŁADÓW LECZNICTWA -> rating 3.3 < 3.5
	✗ SAMODZIELNY PUBL ZESP ZAKŁ LECZNICTWA OTWARTEGO WA -> rating 3.3 < 3.5
	✗ SAMODZIELNY PUBL ZESP ZAKŁ LECZNICTWA OTWARTEGO WA -> rating 3.3 < 3.5
	✗ CENTERMED WARSZAWA SP Z O. O - PORADNIA LEKARZA PO -> rating 3.2 < 3.5
	✗ SAMODZIELNY ZESPÓŁ PUBLICZNYCH ZAKŁADÓW LECZNICTWA -> rating 3.1 < 3.5
	✗ SAMODZIELNY ZESPÓŁ PUBLICZNYCH ZAKŁADÓW LECZNICTWA -> rating 3.1 < 3.5
	✗ SAMODZIELNY PUBL ZESP ZAKŁ LECZNICTWA OTWARTEGO WA -> rating 3.0 < 3.5
